In [ ]:
#
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#


# Health Insurance Risk Analysis using BigQuery ML

This notebook performs actuarial analysis on historical customer and claims data stored in BigQuery, leveraging **BigQuery ML (BQML)** for model training.

**Objectives:**
1. Connect to BigQuery and explore historical customer and claims data.
2. Train a Linear Regression model **using BQML** to estimate the `risk_score` for claims based on historical data.
3. Evaluate the BQML model directly within BigQuery.
4. Demonstrate prediction using the trained BQML model.

## 0. Pre-requisities

Install dependencies and restart the kernel.

In [ ]:
### Install Dependencies
! pip install google google.cloud seaborn google-cloud-bigquery db-dtypes google-cloud-bigquery-storage

## 1. Setup and Data Loading

Import necessary libraries and configure the BigQuery client. Ensure you have authenticated with Google Cloud (e.g., by running `gcloud auth application-default login` in your terminal).

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import google.auth

# --- Configuration ---
# Capture dynamic runtime environment variables injected via kubectl set env
PROJECT_ID = os.environ.get('PROJECT_ID') 
DATASET_ID = os.environ.get('DATASET_ID', 'next_demo_health_insurance_ds')
CUSTOMERS_TABLE = 'historical_customers'
CLAIMS_TABLE = 'historical_claims'
BQML_MODEL_NAME = 'claim_risk_linear_reg_model' # Name for the BQML model
LOCATION = os.environ.get('REGION', 'u-germany-northeast1')

UNIVERSE_DOMAIN = os.environ.get('UNIVERSE_DOMAIN', 'apis-berlin-build0.goog')
os.environ['GOOGLE_CLOUD_UNIVERSE_DOMAIN'] = UNIVERSE_DOMAIN

# Construct full table and model IDs
customers_table_id = f"{PROJECT_ID}.{DATASET_ID}.{CUSTOMERS_TABLE}"
claims_table_id = f"{PROJECT_ID}.{DATASET_ID}.{CLAIMS_TABLE}"
model_id = f"{PROJECT_ID}.{DATASET_ID}.{BQML_MODEL_NAME}"

# --- Authentication Credentials ---
# Use native Workload Identity credentials
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])

# --- BigQuery Client ---
try:
    # Specify location if your dataset/processing needs it
    client = bigquery.Client(credentials=credentials, project=PROJECT_ID, location=LOCATION)
    print(f"✅ BigQuery client initialized for project: {client.project} in location: {LOCATION}")
except Exception as e:
    print(f"Error creating BigQuery client: {e}")
    print("Please ensure you have authenticated, the project ID is correct, and the location matches your dataset.")
    client = None 

# --- Helper function to run BQ queries ---
def run_bq_query(sql: str, job_config=None) -> pd.DataFrame:
    """Runs a BigQuery query and returns the results as a pandas DataFrame."""
    if not client:
        print("BigQuery client not initialized.")
        return None
    try:
        print(f"Running query:\n{sql[:300]}...") # Print start of query
        query_job = client.query(sql, job_config=job_config)
        # Wait for the job to complete, especially for DDL like CREATE MODEL
        results = query_job.result() 
        print(f"Query job {query_job.job_id} finished.")
        # For SELECT queries, fetch results to DataFrame
        if query_job.statement_type == 'SELECT':
             df = results.to_dataframe()
             print(f"Fetched {len(df)} rows.")
             return df
        else:
            print(f"Statement type {query_job.statement_type} executed successfully.")
            return None # No DataFrame for non-SELECT statements
    except Exception as e:
        print(f"Error running BigQuery query: {e}")
        # Potentially print more details from the error object if available
        if hasattr(e, 'errors'):
             print(f"BQ Errors: {e.errors}")
        return None

## 2. Exploratory Data Analysis (Claims Data Sample)

Load the historical claims data for EDA. We do this by executing a SELECT query on BigQuery.

In [6]:
# SQL Query for EDA sample
sql_claims_sample = f"""
SELECT 
    risk_score, 
    patient_age,
    amount_billed,
    processing_time_days,
    public_insurance_base,
    mutuelle_coverage,
    customer_responsibility,
    documentation_count,
    patient_gender,
    provider_type,
    provider_specialty,
    service_category,
    plan_tier,
    is_documentation_complete,
    is_provider_flagged,
    is_amount_unusual,
    is_service_unusual
FROM `{claims_table_id}` 
WHERE risk_score IS NOT NULL
LIMIT 10000 -- Limit sample size
"""

# Fetch sample data for EDA
df_claims_sample = run_bq_query(sql_claims_sample)

# Display basic info for the sample
if df_claims_sample is not None:
    print("\n--- Claims Sample DataFrame Info ---")
    df_claims_sample.info()
    print(df_claims_sample.head())

Perform basic analysis on the sampled claims data.

In [8]:
if df_claims_sample is not None:
    print("\n--- Claims Sample Data Description ---")
    numeric_cols_claims = df_claims_sample.select_dtypes(include=np.number).columns
    print(df_claims_sample[numeric_cols_claims].describe())

    print("\n--- Missing Values in Claims Sample Data ---")
    print(df_claims_sample.isnull().sum())

    # --- Visualizations ---
    plt.style.use('seaborn-v0_8-whitegrid')

    # Distribution of Risk Score (Target Variable)
    plt.figure(figsize=(10, 6))
    sns.histplot(df_claims_sample['risk_score'], kde=True, bins=30)
    plt.title('Distribution of Claim Risk Score (Sample)')
    plt.xlabel('Risk Score')
    plt.ylabel('Frequency')
    plt.show()
    
    # Distribution of Amount Billed
    plt.figure(figsize=(10, 6))
    sns.histplot(np.log1p(df_claims_sample['amount_billed'].astype(float)), kde=True, bins=50) 
    plt.title('Distribution of Log(Amount Billed + 1) (Sample)')
    plt.xlabel('Log(Amount Billed + 1)')
    plt.ylabel('Frequency')
    plt.show()

    # Risk Score vs. Patient Age
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_claims_sample, x='patient_age', y='risk_score', alpha=0.5)
    plt.title('Risk Score vs. Patient Age (Sample)')
    plt.xlabel('Patient Age')
    plt.ylabel('Risk Score')
    plt.show()

else:
    print("Claims Sample DataFrame not loaded. Skipping EDA.")

## 3. Risk Score Prediction Model (BQML Linear Regression)

Train a linear regression model using BigQuery ML's `CREATE MODEL` statement. BQML automatically handles feature preprocessing like one-hot encoding for categorical features and scaling for numerical features by default (though explicit control is possible).

In [9]:
# Define the BQML CREATE MODEL query
# Define the BQML CREATE MODEL query with robust casting
sql_create_model = f"""
CREATE OR REPLACE MODEL `{model_id}`
OPTIONS(
    model_type='LINEAR_REG',
    input_label_cols=['risk_score'],
    enable_global_explain=TRUE,
    hparam_tuning_algorithm = 'VIZIER_DEFAULT',
    hparam_tuning_objectives = ['mean_squared_error']
) AS
SELECT 
    -- Target Variable (Assuming risk_score is numeric)
    risk_score, 
    
    -- Features (Numerical - Cast to STRING -> COALESCE -> SAFE_CAST -> COALESCE)
    COALESCE(SAFE_CAST(COALESCE(CAST(patient_age AS STRING), '0') AS INT64), 0) as patient_age,
    COALESCE(SAFE_CAST(COALESCE(CAST(amount_billed AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as amount_billed,
    COALESCE(SAFE_CAST(COALESCE(CAST(processing_time_days AS STRING), '0') AS INT64), 0) as processing_time_days,
    COALESCE(SAFE_CAST(COALESCE(CAST(public_insurance_base AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as public_insurance_base,
    COALESCE(SAFE_CAST(COALESCE(CAST(mutuelle_coverage AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as mutuelle_coverage,
    COALESCE(SAFE_CAST(COALESCE(CAST(customer_responsibility AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as customer_responsibility,
    COALESCE(SAFE_CAST(COALESCE(CAST(documentation_count AS STRING), '0') AS INT64), 0) as documentation_count,
    
    -- Features (Categorical)
    COALESCE(patient_gender, 'Unknown') as patient_gender,
    COALESCE(provider_type, 'Unknown') as provider_type,
    COALESCE(provider_specialty, 'Unknown') as provider_specialty,
    COALESCE(service_category, 'Unknown') as service_category,
    COALESCE(plan_tier, 'Unknown') as plan_tier,
    
    -- Features (Boolean)
    COALESCE(is_documentation_complete, FALSE) as is_documentation_complete,
    COALESCE(is_provider_flagged, FALSE) as is_provider_flagged,
    COALESCE(is_amount_unusual, FALSE) as is_amount_unusual,
    COALESCE(is_service_unusual, FALSE) as is_service_unusual
    
FROM 
    `{claims_table_id}`
WHERE 
    risk_score IS NOT NULL 
"""

# Define a function to run the query (replace with your actual execution logic)
def run_bq_query(sql, client=None):
    """Runs a BigQuery query."""
    if client is None:
        # Initialize client if not provided (adjust project/location if needed)
        client = bigquery.Client() 
    try:
        print(f"Running query:{sql[:150]}...") # Print start of query
        # Ensure location matches your dataset/error message if necessary
        # job_config = bigquery.QueryJobConfig(default_dataset=f"{client.project}.your_dataset") 
        query_job = client.query(sql) #, location="u-france-east1")  # API request, specify location if needed
        query_job.result()  # Waits for the job to complete
        print("Query finished successfully.")
    except Exception as e:
        print(f"Error running BigQuery query: {e}")
        # Optional: print more details for debugging
        if hasattr(e, 'errors'):
             print(f"BQ Errors: {e.errors}")

# Execute the CREATE MODEL query
print(f"\nCreating/Replacing BQML model: {model_id}...")
run_bq_query(sql_create_model, client) 
print("BQML model training job submission attempt finished.")

## 4. Evaluate BQML Model

Use the `ML.EVALUATE` function to get performance metrics for the trained model. BQML automatically uses a holdout set (or the specified split) for evaluation if not evaluating on specific data.

In [10]:
sql_evaluate_model = f"""
SELECT * 
FROM ML.EVALUATE(MODEL `{model_id}`)
"""

print("\nEvaluating BQML model...")
query_job = client.query(sql_evaluate_model)
df_evaluation = query_job.to_dataframe()

if df_evaluation is not None:
    print("\n--- BQML Model Evaluation Metrics ---")
    print(df_evaluation)

## 5. Feature Importance (Global Explainability)

If `enable_global_explain=TRUE` was set during training, we can view feature attributions.

In [11]:
sql_feature_importance = f"""
SELECT * 
FROM ML.GLOBAL_EXPLAIN(MODEL `{model_id}`)
"""

print("\nFetching BQML model feature importance...")
query_job = client.query(sql_feature_importance)
df_feature_importance = query_job.to_dataframe()

if df_feature_importance is not None:
    print("\n--- BQML Model Feature Importance ---")
    # Sort by absolute attribution for clarity
    df_feature_importance['abs_attribution'] = df_feature_importance['attribution'].abs()
    print(df_feature_importance.sort_values(by='abs_attribution', ascending=False).drop(columns=['abs_attribution']))
    
    # Simple bar plot
    plt.figure(figsize=(10, 8))
    # Take top N features
    top_n_features = 15
    imp_plot = df_feature_importance.sort_values(by='abs_attribution', ascending=False).head(top_n_features)
    sns.barplot(x='attribution', y='feature', data=imp_plot, palette='viridis', hue='feature', legend=False)
    plt.title(f'Top {top_n_features} Feature Attributions (Global Explain)')
    plt.xlabel('Attribution')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Could not retrieve feature importance. Ensure 'enable_global_explain=TRUE' was set during training.")

## 6. Predict with BQML Model

Use the `ML.PREDICT` function to get risk score predictions for new or existing claims data directly in BigQuery.

In [12]:
sql_predict = f"""
SELECT
  claim_id,
  risk_score, -- Actual score for comparison
  predicted_risk_score -- Select the FLOAT64 column directly
FROM
  ML.PREDICT(MODEL `{model_id}`, (
    SELECT 
        claim_id,
        risk_score, -- Include actual for comparison
        
        -- Apply the EXACT SAME robust casting/preprocessing as in CREATE MODEL
        COALESCE(SAFE_CAST(COALESCE(CAST(patient_age AS STRING), '0') AS INT64), 0) as patient_age,
        COALESCE(SAFE_CAST(COALESCE(CAST(amount_billed AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as amount_billed,
        COALESCE(SAFE_CAST(COALESCE(CAST(processing_time_days AS STRING), '0') AS INT64), 0) as processing_time_days,
        COALESCE(SAFE_CAST(COALESCE(CAST(public_insurance_base AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as public_insurance_base,
        COALESCE(SAFE_CAST(COALESCE(CAST(mutuelle_coverage AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as mutuelle_coverage,
        COALESCE(SAFE_CAST(COALESCE(CAST(customer_responsibility AS STRING), '0') AS NUMERIC), CAST(0 AS NUMERIC)) as customer_responsibility,
        COALESCE(SAFE_CAST(COALESCE(CAST(documentation_count AS STRING), '0') AS INT64), 0) as documentation_count,
        
        -- Categorical and Boolean features should match CREATE MODEL as well (likely already correct)
        COALESCE(patient_gender, 'Unknown') as patient_gender,
        COALESCE(provider_type, 'Unknown') as provider_type,
        COALESCE(provider_specialty, 'Unknown') as provider_specialty,
        COALESCE(service_category, 'Unknown') as service_category,
        COALESCE(plan_tier, 'Unknown') as plan_tier,
        COALESCE(is_documentation_complete, FALSE) as is_documentation_complete,
        COALESCE(is_provider_flagged, FALSE) as is_provider_flagged,
        COALESCE(is_amount_unusual, FALSE) as is_amount_unusual,
        COALESCE(is_service_unusual, FALSE) as is_service_unusual
        
    FROM `{claims_table_id}`
    WHERE risk_score IS NOT NULL -- Predict on the same data used for eval/train for demo
    LIMIT 1000 -- Limit prediction sample size
    )
  )
"""

# The rest of your Python code for executing the query and plotting remains the same:

print("\nRunning BQML prediction...")
try:
    query_job = client.query(sql_predict) #, location="u-france-east1") # Specify location if needed
    df_predictions = query_job.to_dataframe()

    if df_predictions is not None and not df_predictions.empty:
        print("\n--- Sample BQML Predictions ---")
        print(df_predictions.head(10))

        # --- Visualization: Predicted vs Actual (BQML) ---
        if 'risk_score' in df_predictions.columns and 'predicted_risk_score' in df_predictions.columns:
            plt.figure(figsize=(10, 6))
            plt.scatter(df_predictions['risk_score'], df_predictions['predicted_risk_score'], alpha=0.5)
            
            valid_actual = df_predictions['risk_score'].dropna()
            valid_predicted = df_predictions['predicted_risk_score'].dropna()
            if not valid_actual.empty and not valid_predicted.empty:
                 min_val = min(valid_actual.min(), valid_predicted.min())
                 max_val = max(valid_actual.max(), valid_predicted.max())
                 plt.plot([min_val, max_val], [min_val, max_val], '--r', linewidth=2, label='Ideal Prediction')
                 plt.legend()

            plt.xlabel('Actual Risk Score')
            plt.ylabel('Predicted Risk Score (BQML)')
            plt.title('Actual vs. Predicted Risk Score (BQML Predictions Sample)')
            plt.grid(True)
            plt.show()
        else:
            print("Error: 'risk_score' or 'predicted_risk_score' column not found in results.")

    elif df_predictions is not None:
         print("Prediction query ran successfully but returned no results.")
    else:
         print("Prediction query failed to return a DataFrame.")

except Exception as e:
    print(f"An error occurred during prediction or processing: {e}")
    if hasattr(e, 'errors'):
         print(f"BQ Errors: {e.errors}")

## 7. Conclusion

This notebook demonstrated connecting to BigQuery, performing basic EDA, and training, evaluating, and predicting with a linear regression model using **BigQuery ML**. This approach keeps computation and data within BigQuery, which is efficient for large datasets.

**BQML automatically handles:**
*   Basic feature preprocessing (scaling, encoding).
*   Data splitting for evaluation (by default).
*   Hyperparameter tuning (if enabled).

**Next Steps could include:**
*   Trying different BQML model types (e.g., `BOOSTED_TREE_REGRESSOR`).
*   More explicit feature engineering within the `CREATE MODEL` query using SQL functions.
*   Using `ML.FEATURE_INFO` to understand how BQML preprocessed features.
*   Deploying the BQML model for batch or online predictions.